# ThermoRG-NN Phase A — Full Theory Validation Pipeline
## Training + TAS Metrics + d_manifold + Publication Plots

**断点续传**: 每个架构独立保存，CSV 存在则跳过；finally 块保证结果总是推回 GitHub。

---

In [ ]:
import os, sys, subprocess
from kaggle_secrets import UserSecretsClient

## ▶️ Step 1: Environment & Code Clone

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url}
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Code cloned and identity configured")

In [ ]:
!pip install -q torch torchvision scipy matplotlib
sys.path.insert(0, "/kaggle/working/ThermoRG-NN/src")
sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
print("Environment ready")

## ▶️ Step 2: Smoke Test

No GPU, no real data — catches import errors, d_manifold computation, JSON/CSV writing. Burns 0 GPU quota.

In [ ]:
!python3 experiments/phase_a/smoke_test.py

## ▶️ Step 3: Training with Smart Checkpoint Resume

Trains all 15 architectures (or skips those with existing CSV). try/except/finally ensures partial results are always pushed.

In [ ]:
try:
    from experiments.lift_test import train
    print("Starting training with checkpoint resume...")
    train.train_phase_a(
        output_dir="experiments/lift_test/results",
        architectures=None  # train all 15
    )
    print("\n🎉 All architectures trained successfully!")
except Exception as e:
    print(f"\n🚨 Training exception: {e}")
    print("Partial results are safe on disk. Will push below.")
finally:
    print("\n📢 [finally] Pushing all completed results to GitHub...")
    results_dir = "experiments/lift_test/results"
    cmds = [
        f"git add {results_dir}/*/*.csv {results_dir}/*/*.pt 2>/dev/null || true",
        'git commit -m "Data: incremental save - training partial results" || true',
        'git push origin main 2>&1 || git push origin master 2>&1 || true'
    ]
    for cmd in cmds:
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                             cwd="/kaggle/working/ThermoRG-NN")
        if res.returncode != 0 and "nothing to commit" not in res.stdout and "nothing to commit" not in res.stderr:
            print(f"  WARN: {res.stderr[-200:]}")
        else:
            print(f"  OK: {cmd[:65]}")
    print("Push complete.")

## ▶️ Step 4: Load Training Results

Reads best accuracy from completed training CSVs. Safe to re-run — just reads, no side effects.

In [ ]:
import glob, json, os, csv

PHASE_A_RESULTS_DIR = "experiments/phase_a/results"
os.makedirs(PHASE_A_RESULTS_DIR, exist_ok=True)

csv_files = sorted(glob.glob("experiments/lift_test/results/*/phase_a_metrics.csv"))
print(f"Found {len(csv_files)} training CSV(s):")

actual_results = {}
for csv_path in csv_files:
    arch = csv_path.split("/")[-2]
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    # Support: 'acc', 'val_acc', or 'test_acc' column
    acc_col = "val_acc" if "val_acc" in rows[0] else ("test_acc" if "test_acc" in rows[0] else "acc")
    best = max(float(r[acc_col]) for r in rows)
    actual_results[arch] = {"best_acc": float(best)}
    print(f"  {arch}: best_{acc_col}={best:.2f}%")

print(f"\nTotal: {len(actual_results)} architectures with results")

with open(f"{PHASE_A_RESULTS_DIR}/training_results.json", "w") as f:
    json.dump(actual_results, f, indent=2)
print(f"Saved to {PHASE_A_RESULTS_DIR}/training_results.json")

## ▶️ Step 5: TAS Profiling (GPU)

Computes d_manifold + alpha/J_topo metrics for all 15 architectures + generates 3 publication plots. Reads from local training results — safe to re-run with --use-fake if results are missing.

In [ ]:
r = subprocess.run(
    [
        "python3",
        "experiments/phase_a/phase_a_analysis.py",
        "--device", "cuda",
        "--n-samples", "5000",
        "--n-per-arch", "500",
        "--actual-results", f"{PHASE_A_RESULTS_DIR}/training_results.json",
        "--output-dir", PHASE_A_RESULTS_DIR,
    ],
    capture_output=True,
    text=True,
)
print("=== STDOUT ===")
print(r.stdout[-4000:] if r.stdout else "(empty)")
print("\n=== STDERR ===")
print(r.stderr[-1000:] if r.stderr else "(empty)")
print(f"\nReturn code: {r.returncode}")

## ▶️ Step 6: Push All Results to GitHub

Always runs — even if TAS profiling partially failed. Pushes training CSVs + TAS results + plots.

In [ ]:
TRAIN_DIR = "experiments/lift_test/results"
cmds = [
    # Push training CSVs + model weights
    f"git add {TRAIN_DIR}/*/*.csv {TRAIN_DIR}/*/*.pt 2>/dev/null || true",
    # Push TAS profiling results + plots
    f"git add {PHASE_A_RESULTS_DIR}/*.csv {PHASE_A_RESULTS_DIR}/*.json {PHASE_A_RESULTS_DIR}/*.png 2>/dev/null || true",
    'git commit -m "Phase A complete: training + TAS metrics + plots from Kaggle" || true',
    'git push origin main 2>&1 || git push origin master 2>&1 || true'
]

for cmd in cmds:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                         cwd="/kaggle/working/ThermoRG-NN")
    if res.returncode != 0 and "nothing to commit" not in res.stdout and "nothing to commit" not in res.stderr:
        print(f"WARN [{cmd[:40]}...]: {res.stderr[-200:]}")
    else:
        print(f"OK: {cmd[:70]}")

print("\n🎉 Phase A pipeline complete! Check GitHub for results.")